In [15]:
import pandas as pd
from pathlib import Path

print("Netflix Customer Intelligence Copilot")
print("AI layer started")

Netflix Customer Intelligence Copilot
AI layer started


In [16]:
data_path = Path("../data/cleaned")

customers = pd.read_csv(
    data_path / "customers.csv"
)

risk_scores = pd.read_csv(
    data_path / "customer_churn_risk.csv"
)

feedback = pd.read_csv(
    data_path / "customer_feedback.csv"
)

print("Customers:", customers.shape)
print("Risk scores:", risk_scores.shape)
print("Feedback:", feedback.shape)

Customers: (8000, 11)
Risk scores: (6571, 3)
Feedback: (5046, 6)


In [17]:
customer_intelligence = customers.merge(
    risk_scores,
    on="customer_id",
    how="left"
)

print("Customer intelligence shape:", customer_intelligence.shape)

print("\nAvailable columns:")
print(customer_intelligence.columns.tolist())

print("\nFirst 5 customers:")
print(
    customer_intelligence[
        [
            "customer_id",
            "customer_segment",
            "acquisition_channel",
            "churn_probability",
            "risk_level"
        ]
    ].head()
)

Customer intelligence shape: (8000, 13)

Available columns:
['customer_id', 'first_name', 'last_name', 'age', 'gender', 'country', 'city', 'registration_date', 'acquisition_channel', 'customer_segment', 'preferred_language', 'churn_probability', 'risk_level']

First 5 customers:
  customer_id customer_segment       acquisition_channel  churn_probability  \
0  CUST000744       Individual            Organic Search           0.146667   
1  CUST001502           Family                  Referral           0.470000   
2  CUST005383          Student               Paid Social           0.083333   
3  CUST004708       Individual  Partner Bundle (Telecom)                NaN   
4  CUST005877          Student            Email Campaign           0.230000   

  risk_level  
0        Low  
1     Medium  
2        Low  
3        NaN  
4     Medium  


In [18]:
customer_features = pd.read_csv(
    data_path / "customer_features.csv"
)

print("Customer features shape:", customer_features.shape)

print("\nAvailable features:")
print(customer_features.columns.tolist())

Customer features shape: (8000, 35)

Available features:
['customer_id', 'first_name', 'last_name', 'age', 'gender', 'country', 'city', 'registration_date', 'acquisition_channel', 'customer_segment', 'preferred_language', 'tenure_days', 'total_watch_time_minutes', 'total_viewing_sessions', 'average_completion_percentage', 'average_rating', 'feedback_count', 'support_ticket_count', 'average_support_satisfaction', 'has_support_satisfaction', 'successful_payment_count', 'total_successful_payment_amount', 'failed_payment_count', 'subscription_count', 'has_plan_change', 'is_active_subscription', 'average_watch_time_per_session', 'unique_titles_watched', 'active_viewing_days', 'current_plan', 'payment_failure_rate', 'favorite_genre', 'unique_genres_watched', 'current_subscription_tenure_days', 'has_failed_payment']


In [19]:
# Merge engineered customer features with churn risk
ai_data = customer_features.merge(
    risk_scores,
    on="customer_id",
    how="left"
)

# Keep only useful customer intelligence columns
context_columns = [
    "customer_id",
    "age",
    "gender",
    "country",
    "city",
    "acquisition_channel",
    "customer_segment",
    "preferred_language",
    "current_plan",
    "favorite_genre",
    "tenure_days",
    "total_watch_time_minutes",
    "total_viewing_sessions",
    "average_completion_percentage",
    "average_rating",
    "feedback_count",
    "support_ticket_count",
    "average_support_satisfaction",
    "successful_payment_count",
    "total_successful_payment_amount",
    "failed_payment_count",
    "payment_failure_rate",
    "subscription_count",
    "current_subscription_tenure_days",
    "churn_probability",
    "risk_level"
]

ai_data = ai_data[context_columns]

print("AI customer context shape:", ai_data.shape)
print("\nColumns:", len(ai_data.columns))
print("\nSample:")
print(ai_data.head())

AI customer context shape: (8000, 26)

Columns: 26

Sample:
  customer_id   age  gender         country        city  \
0  CUST000744  21.0  Female  United Kingdom   Liverpool   
1  CUST001502  31.0    Male          Canada    Montreal   
2  CUST005383  40.0    Male            U.K.       Leeds   
3  CUST004708  27.0    Male  United Kingdom  Manchester   
4  CUST005877  36.0    Male       Indonesia     Jakarta   

        acquisition_channel customer_segment preferred_language current_plan  \
0            Organic Search       Individual            English       PLAN03   
1                  Referral           Family             French       PLAN01   
2               Paid Social          Student            English       PLAN01   
3  Partner Bundle (Telecom)       Individual              Hindi       PLAN02   
4            Email Campaign          Student             French       PLAN03   

  favorite_genre  ...  support_ticket_count  average_support_satisfaction  \
0         Action  ...      

In [20]:
# Prepare feedback lookup by customer
feedback_lookup = (
    feedback
    .groupby("customer_id")["feedback_text"]
    .apply(list)
    .to_dict()
)

def get_customer_context(customer_id):
    customer = ai_data[
        ai_data["customer_id"] == customer_id
    ]

    if customer.empty:
        return None

    customer_info = customer.iloc[0].to_dict()

    customer_feedback = feedback_lookup.get(
        customer_id,
        []
    )

    return {
        "customer_profile": customer_info,
        "feedback": customer_feedback
    }

print("Customer context function ready.")

# Test
test_customer = ai_data["customer_id"].iloc[0]

context = get_customer_context(test_customer)

print("\nTest customer:", test_customer)
print("Profile fields:", len(context["customer_profile"]))
print("Feedback entries:", len(context["feedback"]))

Customer context function ready.

Test customer: CUST000744
Profile fields: 26
Feedback entries: 1


In [22]:
import pandas as pd
import numpy as np

def local_copilot(customer_id, question):
    customer = ai_data[
        ai_data["customer_id"] == customer_id
    ]

    if customer.empty:
        return "Customer not found."

    c = customer.iloc[0]

    question = question.lower()

    # Customer summary
    if "summary" in question or "profile" in question:
        return f"""
Customer {customer_id} is a {c['customer_segment']} customer
on the {c['current_plan']} plan.

Engagement:
- Watch time: {c['total_watch_time_minutes']:.0f} minutes
- Viewing sessions: {c['total_viewing_sessions']:.0f}
- Completion rate: {c['average_completion_percentage']:.1f}%
- Favorite genre: {c['favorite_genre']}

Payments:
- Failed payments: {c['failed_payment_count']:.0f}
- Payment failure rate: {c['payment_failure_rate']:.1%}

Support:
- Tickets: {c['support_ticket_count']:.0f}

Churn risk:
- Probability: {c['churn_probability']:.1%}
- Risk level: {c['risk_level']}

This is a risk signal, not a certainty of churn.
"""

    # Churn risk explanation
    if "risk" in question or "churn" in question:
        reasons = []

        if pd.notna(c["payment_failure_rate"]) and c["payment_failure_rate"] > 0.05:
            reasons.append("elevated payment failure rate")

        if c["total_viewing_sessions"] < 5:
            reasons.append("low viewing activity")

        if c["average_completion_percentage"] < 60:
            reasons.append("low content completion")

        if c["support_ticket_count"] >= 2:
            reasons.append("multiple support interactions")

        if pd.notna(c["average_support_satisfaction"]) and c["average_support_satisfaction"] < 3:
            reasons.append("low support satisfaction")

        if not reasons:
            reasons.append("no major negative signal identified from the available features")

        return f"""
Customer {customer_id} has a {c['risk_level']} churn-risk classification
with a model probability of {c['churn_probability']:.1%}.

Potential contributing signals:
- """ + "\n- ".join(reasons) + """

These are associations in the available customer data, not proof of causation.
"""

    return """
I can currently answer:
1. Customer profile/summary
2. Churn risk and possible reasons

Example:
local_copilot("CUST000744", "Give me a summary")
local_copilot("CUST000744", "Why is this customer high risk?")
"""


print("Free local Copilot engine ready.")

Free local Copilot engine ready.


In [23]:
local_copilot(
    "CUST000744",
    "Give me a summary of this customer"
)

'\nCustomer CUST000744 is a Individual customer\non the PLAN03 plan.\n\nEngagement:\n- Watch time: 266 minutes\n- Viewing sessions: 5\n- Completion rate: 70.2%\n- Favorite genre: Action\n\nPayments:\n- Failed payments: 2\n- Payment failure rate: 13.3%\n\nSupport:\n- Tickets: 0\n\nChurn risk:\n- Probability: 14.7%\n- Risk level: Low\n\nThis is a risk signal, not a certainty of churn.\n'

In [24]:
def copilot_business_query(question):
    q = question.lower()

    # 1. High-risk customers
    if "high-risk" in q or "high risk" in q:
        result = ai_data[
            ai_data["risk_level"] == "High"
        ].copy()

        if "payment" in q:
            result = result[
                result["failed_payment_count"] > 0
            ]

        return result[
            [
                "customer_id",
                "current_plan",
                "failed_payment_count",
                "payment_failure_rate",
                "churn_probability",
                "risk_level"
            ]
        ].sort_values(
            "churn_probability",
            ascending=False
        ).head(20)

    # 2. Plan with highest churn
    if "plan" in q and "churn" in q:
        plan_churn = (
            customer_features
            .groupby("current_plan")
            .agg(
                customers=("customer_id", "count"),
                avg_churn_risk=("customer_id", "count")
            )
            .reset_index()
        )

        churn_labels_data = pd.read_csv(
            data_path / "churn_labels.csv"
        )

        plan_data = customer_features[
            ["customer_id", "current_plan"]
        ].merge(
            churn_labels_data[
                ["customer_id", "churn_flag"]
            ],
            on="customer_id",
            how="left"
        )

        return (
            plan_data
            .groupby("current_plan")
            .agg(
                customers=("customer_id", "count"),
                churned_customers=("churn_flag", "sum")
            )
            .assign(
                churn_rate=lambda x:
                x["churned_customers"] / x["customers"]
            )
            .sort_values("churn_rate", ascending=False)
        )

    # 3. Negative complaints
    if "negative" in q and ("complaint" in q or "feedback" in q):
        negative_feedback = feedback[
            feedback["sentiment"].str.lower() == "negative"
        ]

        return negative_feedback[
            [
                "customer_id",
                "rating",
                "feedback_text"
            ]
        ].head(20)

    return """
I can answer these business questions:

1. Show me high-risk customers.
2. Show me high-risk customers with payment failures.
3. Which plan has the highest churn?
4. What are the main negative complaints?

For individual customers:
local_copilot("CUST000744", "Why is this customer high risk?")
"""


print("Business Copilot queries ready.")

Business Copilot queries ready.


In [25]:
copilot_business_query(
    "Show me high-risk customers with payment failures"
)

,customer_id,current_plan,failed_payment_count,payment_failure_rate,churn_probability,risk_level
7566,CUST007445,PLAN02,1.0,0.066667,1.0,High
7569,CUST003355,PLAN01,3.0,0.200000,1.0,High
7440,CUST006967,PLAN02,2.0,0.166667,1.0,High
1264,CUST000352,PLAN01,3.0,0.200000,1.0,High
1166,CUST005774,PLAN01,1.0,0.066667,1.0,High
7249,CUST002783,PLAN01,1.0,0.333333,1.0,High
7229,CUST005306,PLAN01,3.0,0.200000,1.0,High
846,CUST000322,PLAN03,1.0,0.142857,1.0,High
974,CUST004871,PLAN03,2.0,0.133333,1.0,High
986,CUST006685,PLAN01,4.0,0.266667,1.0,High


In [26]:
def retention_action(customer_id):
    customer = ai_data[
        ai_data["customer_id"] == customer_id
    ]

    if customer.empty:
        return "Customer not found."

    c = customer.iloc[0]
    actions = []

    if c["failed_payment_count"] > 0:
        actions.append("Contact the customer about payment issues and provide a payment-method update option.")

    if c["total_viewing_sessions"] < 5:
        actions.append("Send personalized content recommendations to increase engagement.")

    if c["average_completion_percentage"] < 60:
        actions.append("Recommend shorter or highly relevant content to improve completion.")

    if pd.notna(c["average_support_satisfaction"]) and c["average_support_satisfaction"] < 3:
        actions.append("Prioritize a support follow-up because satisfaction is low.")

    if not actions:
        actions.append("Continue engagement monitoring and personalized content recommendations.")

    return f"""
Retention recommendations for {customer_id}:

- """ + "\n- ".join(actions)


print(retention_action("CUST000744"))


Retention recommendations for CUST000744:

- Contact the customer about payment issues and provide a payment-method update option.


In [27]:
%pip install -U google-genai

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 4.8 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ------------------------------ --------- 1.6/2.0 MB 7.6 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 6.7 MB/s  0:00:00

  Attempting uninstall: pydantic-core

    Found existing installation: pydantic_core 2.41.5

    Uninstalling pydantic_core-2.41.5:

      Successfully uninstalled pydantic_core-2.41.5

   ---------------------------------------- 0/4 [pydantic-core]
   ---------------------------------------- 0/4 [pydantic-core]
   ---------------------------------------- 0/4 [pydantic-core]
   ---------------------------------------- 0/4 [pydantic-core]
   ---------------------------------------- 0/4 [pydantic-core]
   ---------------------------------------- 0/4 [pydantic-core]
   ---------------------------------------- 0/4 [pydantic-core]
   -

  You can safely remove it manually.


In [28]:
import getpass
from google import genai

gemini_key = getpass.getpass("Enter your Gemini API key: ")

gemini_client = genai.Client(
    api_key=gemini_key
)

print("Gemini AI connected.")

Gemini AI connected.


In [33]:
def ask_gemini(customer_id, question):
    context = get_customer_context(customer_id)

    if context is None:
        return "Customer not found."

    prompt = f"""
You are a Netflix Customer Intelligence Copilot.

Answer the question using ONLY the customer data provided below.
Do not invent information.

CUSTOMER PROFILE:
{context["customer_profile"]}

CUSTOMER FEEDBACK:
{context["feedback"]}

USER QUESTION:
{question}

Instructions:
- Give a concise business-focused answer.
- Explain important risk signals when relevant.
- Recommend practical retention actions when appropriate.
- Clearly state when the available data is insufficient.
- Treat churn probability as a model risk score, not certainty.
"""

    response = gemini_client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    return response.text


print("Generative AI Copilot ready.")

Generative AI Copilot ready.


In [34]:
ask_gemini(
    "CUST000744",
    "Give me a short summary of this customer and explain any important churn risk signals."
)

'### Customer Summary: CUST000744\nThis is a long-tenured (2,584 days) individual subscriber based in Liverpool, UK, currently on a PLAN03 subscription. While the customer has been with Netflix for over seven years, their engagement is exceptionally low, with only 266 total watch minutes and five viewing sessions recorded. They show a preference for Action content and have provided neutral feedback (3.0 rating) regarding a desire for more regional language variety.\n\n### Churn Risk Signals\n*   **Low Engagement:** The primary risk factor is the extremely low volume of activity relative to the tenure. A customer with over seven years of tenure but only five total viewing sessions indicates they are a "zombie" subscriber who may be overlooking the service.\n*   **Payment Friction:** The customer has a payment failure rate of 13.3%, which, while not currently critical, is a potential point of involuntary churn if billing issues persist.\n*   **Model Risk Score:** The churn probability is